# Importing the Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Salaries.csv")

In [3]:
df.head()

,Unnamed: 0,rank,discipline,yrs.since.phd,yrs.service,sex,salary
0,1,Prof,B,19.0,18.0,Male,139750
1,2,Prof,B,20.0,16.0,Male,173200
2,3,AsstProf,B,4.0,3.0,Male,79750
3,4,Prof,B,45.0,39.0,Male,115000
4,5,Prof,B,40.0,41.0,Male,141500


# Dropping the Unnecessary columns


In [5]:
df.drop("Unnamed: 0", axis = 1, inplace=True)
df.head()

,rank,discipline,yrs.since.phd,yrs.service,sex,salary
0,Prof,B,19.0,18.0,Male,139750
1,Prof,B,20.0,16.0,Male,173200
2,AsstProf,B,4.0,3.0,Male,79750
3,Prof,B,45.0,39.0,Male,115000
4,Prof,B,40.0,41.0,Male,141500


# Missing Value Treatment

Checking for Null Vales

In [6]:
df.isnull().sum()

rank             2
discipline       0
yrs.since.phd    1
yrs.service      1
sex              1
salary           0
dtype: int64

In [12]:
# Seperating the numerical and categorical columns
numeric_cols = df.select_dtypes(include = np.number).columns
categorical_cols = df.select_dtypes(exclude = np.number).columns
print(numeric_cols)
print(categorical_cols)

Index(['yrs.since.phd', 'yrs.service', 'salary'], dtype='object')
Index(['rank', 'discipline', 'sex'], dtype='object')


In [13]:
df.dtypes

rank              object
discipline        object
yrs.since.phd    float64
yrs.service      float64
sex               object
salary             int64
dtype: object

### Filling Numeric columns with median


In [14]:
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

### Filling categorical columns with mode

In [15]:
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [16]:
df.isnull().sum()

rank             0
discipline       0
yrs.since.phd    0
yrs.service      0
sex              0
salary           0
dtype: int64

### Dropping of Rows

In [17]:
df.dropna(inplace=True)

### Filling with Simple Imputer

In [18]:
from sklearn.impute import SimpleImputer
mean_imputer = SimpleImputer(strategy="mean")
df[numeric_cols] = mean_imputer.fit_transform(df[numeric_cols])

In [19]:
cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

In [20]:
df.isnull().sum()

rank             0
discipline       0
yrs.since.phd    0
yrs.service      0
sex              0
salary           0
dtype: int64

### Duplicate Removal

In [22]:
duplicate_count = df.duplicated().sum()
print(duplicate_count)

7


In [24]:
df = df.drop_duplicates()

In [27]:
df.duplicated().sum()

np.int64(0)

In [26]:
df

,rank,discipline,yrs.since.phd,yrs.service,sex,salary
0,Prof,B,19.0,18.0,Male,139750.0
1,Prof,B,20.0,16.0,Male,173200.0
2,AsstProf,B,4.0,3.0,Male,79750.0
3,Prof,B,45.0,39.0,Male,115000.0
4,Prof,B,40.0,41.0,Male,141500.0
...,...,...,...,...,...,...
395,Prof,A,33.0,30.0,Male,103106.0
396,Prof,A,31.0,19.0,Male,150564.0
397,Prof,A,42.0,25.0,Male,101738.0
398,Prof,A,25.0,15.0,Male,95329.0


### Outlier Detection and Treatment

In [28]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR